# P2 training + noise / TTA / high-res ablation

1. Train YOLOv12n with the stride-4 P2 head (`cfg/yolo12n_p2.yaml`).
2. Compare baseline vs P2 on the test split under Gaussian noise (σ = 10 / 20 / 30), TTA, and 960-px inference.

Metrics: mAP@0.5, mAP@0.5:0.95, precision, recall, F1, FNR, FPS. Writes `ablation_outputs/`. An existing `best.pt` is reused unless you delete it.

**Comparison note.** `AUG_PROFILE='strong'` applies only to P2. The published baseline was trained with Ultralytics defaults (`mixup=0`, `copy_paste=0`, `degrees=0`). Reported deltas are **P2 + strong aug vs stock YOLOv12n**, not a pure head ablation. To isolate the head, retrain the baseline with `AUG_PARAMS_STRONG` or P2 with `AUG_PROFILE='default'`.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.chdir('/content/drive/MyDrive/lld-net-pcb-ml')
except ImportError:
    pass

from project_paths import setup, prepare_dataset, sync_run

P = setup()
os.chdir(P.repo_root)
print('repo :', P.repo_root)
print('colab:', P.in_colab)

In [ ]:
import shutil

if shutil.which('nvidia-smi'):
    !nvidia-smi
else:
    print('nvidia-smi not found — CPU / no NVIDIA driver.')

try:
    import google.colab
    %pip -q install ultralytics pandas pyyaml opencv-python matplotlib
except ImportError:
    pass

In [ ]:
import os
import json
import time
import shutil
import random
from pathlib import Path

import yaml
import numpy as np
import cv2
import pandas as pd
from ultralytics import YOLO
import ultralytics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Ultralytics:', ultralytics.__version__)

In [ ]:
REPO_ROOT          = P.repo_root
DRIVE_DATASET_DIR  = P.drive_dataset
DRIVE_DATA_YAML    = P.drive_data_yaml
DRIVE_RUNS_DIR     = P.drive_runs
DRIVE_CFG_DIR      = P.drive_cfg
DRIVE_ABL_DIR      = P.drive_abl

LOCAL_DATASET_DIR  = P.local_dataset
LOCAL_DATA_YAML    = P.local_data_yaml
LOCAL_PROJECT_DIR  = P.local_runs
LOCAL_NOISE_ROOT   = P.local_noise

RUN_BASELINE_NAME  = 'yolo12_pcb_baseline'
BASELINE_BEST      = P.baseline_best
P2_MODEL_CFG       = P.p2_cfg

# 'strong' = augmentation profile used in the report; 'default' = Ultralytics defaults.
AUG_PROFILE        = 'strong'
RUN_P2_NAME        = f'yolo12n_p2_{AUG_PROFILE}'

IMG_SIZE       = 640
BATCH          = 16
EPOCHS_P2      = 100
DEVICE         = 0 if __import__('torch').cuda.is_available() else 'cpu'
WORKERS        = 4
PATIENCE       = 30
SAVE_PERIOD    = 5

NOISE_LEVELS   = [10, 20, 30]   # additive Gaussian sigma in 0-255 space
CONF_THRES     = 0.25
MAX_FPS_IMAGES = 200

# Strong policy: heavy spatial mix + mild colour/exposure jitter.
# flipud=0 because vertical flip is unrealistic for PCBs.
AUG_PARAMS_STRONG = dict(
    mosaic=1.0, mixup=0.10, copy_paste=0.30,
    degrees=5.0, translate=0.10, scale=0.50,
    fliplr=0.50, flipud=0.0,
    hsv_h=0.015, hsv_s=0.70, hsv_v=0.50,
    erasing=0.40,
)
AUG_PARAMS_DEFAULT = dict()

for d in (DRIVE_ABL_DIR, DRIVE_RUNS_DIR, DRIVE_CFG_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not P2_MODEL_CFG.exists():
    raise FileNotFoundError(f'P2 cfg missing: {P2_MODEL_CFG}')
if not BASELINE_BEST.exists():
    raise FileNotFoundError(
        f'Baseline best.pt missing: {BASELINE_BEST}\n'
        'Run train_yolov12_pcb.ipynb first.'
    )

print('BASELINE_BEST:', BASELINE_BEST)
print('P2_MODEL_CFG :', P2_MODEL_CFG)

In [ ]:
# Colab: cache Drive → VM SSD. Local: use PCB_DATA_YOLO/ in place.
prepare_dataset(P)

for split in ['train', 'val', 'test']:
    n_img = len(list((LOCAL_DATASET_DIR / 'images' / split).glob('*')))
    n_lbl = len(list((LOCAL_DATASET_DIR / 'labels' / split).glob('*.txt')))
    print(f'{split:5s} -> images: {n_img:5d}, labels: {n_lbl:5d}')

In [ ]:
# Instantiate from cfg, then partial-transfer matching layers from yolo12n.pt
# (the new P2 path stays randomly initialised).
p2_model = YOLO(str(P2_MODEL_CFG)).load('yolo12n.pt')
det = p2_model.model.model[-1]
print(f'Detect: {type(det).__name__}  nl={getattr(det, "nl", "?")}  nc={getattr(det, "nc", "?")}')

In [ ]:
LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
p2_run_dir       = LOCAL_PROJECT_DIR / RUN_P2_NAME
p2_last_ckpt     = p2_run_dir / 'weights' / 'last.pt'
P2_BEST          = p2_run_dir / 'weights' / 'best.pt'
drive_p2_run_dir = DRIVE_RUNS_DIR / RUN_P2_NAME

# Pull a previous Drive run back into local cache so resume can pick up.
if drive_p2_run_dir.exists() and not p2_run_dir.exists():
    shutil.copytree(drive_p2_run_dir, p2_run_dir)

aug_kwargs = AUG_PARAMS_STRONG if AUG_PROFILE == 'strong' else AUG_PARAMS_DEFAULT

if p2_last_ckpt.exists():
    print('Resuming from:', p2_last_ckpt)
    p2_model = YOLO(str(p2_last_ckpt))
    p2_train = p2_model.train(resume=True)
elif P2_BEST.exists():
    print('Existing checkpoint found — skip training:', P2_BEST)
    print('Delete best.pt if you want to retrain from scratch.')
    p2_model = YOLO(str(P2_BEST))
else:
    p2_train = p2_model.train(
        data        = str(LOCAL_DATA_YAML),
        epochs      = EPOCHS_P2,
        imgsz       = IMG_SIZE,
        batch       = BATCH,
        device      = DEVICE,
        workers     = WORKERS,
        project     = str(LOCAL_PROJECT_DIR),
        name        = RUN_P2_NAME,
        exist_ok    = True,
        pretrained  = True,
        cache       = True,
        verbose     = True,
        seed        = SEED,
        deterministic = True,
        patience    = PATIENCE,
        save_period = SAVE_PERIOD,
        **aug_kwargs,
    )

if not P2_BEST.exists():
    raise FileNotFoundError(f'P2 best.pt not found: {P2_BEST}')
print('P2 best.pt:', P2_BEST)

In [ ]:
sync_run(p2_run_dir, drive_p2_run_dir)

In [ ]:
baseline_model = YOLO(str(BASELINE_BEST))
p2_best_model  = YOLO(str(P2_BEST))

In [ ]:
# Build a separate noisy test split for each sigma (deterministic per-image seed).
def add_gaussian_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    noise = np.random.normal(0.0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def make_noise_dataset(src_dataset: Path, dst_root: Path, sigma: float):
    if dst_root.exists():
        shutil.rmtree(dst_root)
    shutil.copytree(src_dataset, dst_root)
    rng_state = np.random.get_state()
    for p in (dst_root / 'images' / 'test').glob('*'):
        img = cv2.imread(str(p))
        if img is None:
            continue
        np.random.seed(SEED + int(sigma) + sum(ord(c) for c in p.name))
        cv2.imwrite(str(p), add_gaussian_noise(img, sigma))
    np.random.set_state(rng_state)
    yml_path = dst_root / 'data.yaml'
    with open(yml_path, 'r', encoding='utf-8') as f:
        y = yaml.safe_load(f)
    y['path'] = str(dst_root)
    with open(yml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(y, f, sort_keys=False)
    return dst_root

for sigma in NOISE_LEVELS:
    out = LOCAL_NOISE_ROOT / f'sigma_{sigma}'
    make_noise_dataset(LOCAL_DATASET_DIR, out, sigma)
    print(f'sigma={sigma:>3d} -> {out}')

In [ ]:
def metric_row(tag, metrics_obj, fps=None):
    d = metrics_obj.results_dict
    p = d.get('metrics/precision(B)')
    r = d.get('metrics/recall(B)')
    f1 = (2 * p * r) / (p + r) if (p is not None and r is not None and p + r > 0) else None
    return {
        'exp'       : tag,
        'map50'     : d.get('metrics/mAP50(B)'),
        'map50_95'  : d.get('metrics/mAP50-95(B)'),
        'precision' : p,
        'recall'    : r,
        'f1'        : f1,
        'fnr'       : (1.0 - r) if r is not None else None,
        'fps'       : fps,
    }

def measure_fps(model, image_dir: Path, imgsz=640, conf=0.25, max_images=200):
    imgs = sorted([p for p in Path(image_dir).glob('*')
                   if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']])[:max_images]
    if not imgs:
        return None
    # Warm-up call to exclude one-off CUDA / autograd init from the timing.
    _ = model.predict(source=str(imgs[0]), imgsz=imgsz, conf=conf, verbose=False)
    t0 = time.time()
    _ = model.predict(source=[str(p) for p in imgs], imgsz=imgsz, conf=conf, verbose=False)
    return len(imgs) / max(time.time() - t0, 1e-9)

In [ ]:
test_imgs_dir = LOCAL_DATASET_DIR / 'images' / 'test'
baseline_fps = measure_fps(baseline_model, test_imgs_dir, IMG_SIZE, CONF_THRES, MAX_FPS_IMAGES)
p2_fps       = measure_fps(p2_best_model,  test_imgs_dir, IMG_SIZE, CONF_THRES, MAX_FPS_IMAGES)
print(f'Baseline FPS: {baseline_fps:.2f}')
print(f'P2 FPS      : {p2_fps:.2f}')

In [ ]:
def val_clean(model, split):
    return model.val(data=str(LOCAL_DATA_YAML), split=split,
                     imgsz=IMG_SIZE, device=DEVICE, verbose=False)

def val_noise(model, sigma):
    return model.val(data=str(LOCAL_NOISE_ROOT / f'sigma_{sigma}' / 'data.yaml'),
                     split='test', imgsz=IMG_SIZE, device=DEVICE, verbose=False)

rows = [
    metric_row('baseline_val_clean',  val_clean(baseline_model, 'val')),
    metric_row('p2_val_clean',        val_clean(p2_best_model,  'val')),
    metric_row('baseline_test_clean', val_clean(baseline_model, 'test'), fps=baseline_fps),
    metric_row('p2_test_clean',       val_clean(p2_best_model,  'test'), fps=p2_fps),
]
for sigma in NOISE_LEVELS:
    rows.append(metric_row(f'baseline_test_noise_s{sigma}', val_noise(baseline_model, sigma)))
    rows.append(metric_row(f'p2_test_noise_s{sigma}',       val_noise(p2_best_model,  sigma)))

df = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda v: f'{v:.4f}' if isinstance(v, float) else str(v))
df

In [ ]:
(DRIVE_ABL_DIR / 'plots').mkdir(parents=True, exist_ok=True)
csv_path     = DRIVE_ABL_DIR / 'ablation_results.csv'
summary_path = DRIVE_ABL_DIR / 'ablation_summary.json'

df.to_csv(csv_path, index=False)
summary = {
    'seed'             : SEED,
    'img_size'         : IMG_SIZE,
    'batch'            : BATCH,
    'epochs_p2'        : EPOCHS_P2,
    'noise_levels'     : NOISE_LEVELS,
    'baseline_weights' : str(BASELINE_BEST),
    'p2_cfg'           : str(P2_MODEL_CFG),
    'p2_best'          : str(P2_BEST),
    'baseline_fps'     : baseline_fps,
    'p2_fps'           : p2_fps,
    'rows'             : rows,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
sync_run(p2_run_dir, drive_p2_run_dir)
print('CSV :', csv_path)
print('JSON:', summary_path)

In [ ]:
# Inference-time enhancements (no retraining): TTA (augment=True) and 960-pixel inference.
HIGHRES_IMGSZ = 960

def val_kwargs(model, **extra):
    return model.val(data=str(LOCAL_DATA_YAML), split='test',
                     imgsz=IMG_SIZE, device=DEVICE, verbose=False, **extra)

extra_rows = [
    metric_row('baseline_test_clean_tta',                    val_kwargs(baseline_model, augment=True)),
    metric_row('p2_test_clean_tta',                          val_kwargs(p2_best_model,  augment=True)),
    metric_row(f'baseline_test_clean_imgsz{HIGHRES_IMGSZ}',
               baseline_model.val(data=str(LOCAL_DATA_YAML), split='test',
                                  imgsz=HIGHRES_IMGSZ, device=DEVICE, verbose=False)),
    metric_row(f'p2_test_clean_imgsz{HIGHRES_IMGSZ}',
               p2_best_model.val (data=str(LOCAL_DATA_YAML), split='test',
                                  imgsz=HIGHRES_IMGSZ, device=DEVICE, verbose=False)),
]

df_full = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)
df_full.to_csv(csv_path, index=False)

summary['extra_rows']    = extra_rows
summary['highres_imgsz'] = HIGHRES_IMGSZ
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
df_full

In [ ]:
import matplotlib.pyplot as plt

def get_map50(tag):
    r = df[df['exp'] == tag]
    return float(r['map50'].iloc[0]) if len(r) else None

sigmas       = [0] + NOISE_LEVELS
baseline_map = [get_map50('baseline_test_clean')] + [get_map50(f'baseline_test_noise_s{s}') for s in NOISE_LEVELS]
p2_map       = [get_map50('p2_test_clean')]       + [get_map50(f'p2_test_noise_s{s}')       for s in NOISE_LEVELS]

plt.figure(figsize=(7, 4))
plt.plot(sigmas, baseline_map, 'o-', label='Baseline (YOLOv12n)')
plt.plot(sigmas, p2_map,       's-', label='YOLOv12n + P2')
plt.xlabel('Gaussian noise sigma'); plt.ylabel('mAP@0.5 (test)')
plt.title('Noise robustness: baseline vs P2')
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(DRIVE_ABL_DIR / 'plots' / 'mAP50_vs_noise.png', dpi=150)
plt.show()

In [ ]:
def gv(tag, col='map50'):
    s = df_full[df_full['exp'] == tag]
    return float(s[col].iloc[0]) if len(s) else float('nan')

settings = ['clean_640', 'clean_640_TTA', f'clean_{HIGHRES_IMGSZ}']
b_vals = [gv('baseline_test_clean'), gv('baseline_test_clean_tta'),
          gv(f'baseline_test_clean_imgsz{HIGHRES_IMGSZ}')]
p_vals = [gv('p2_test_clean'),       gv('p2_test_clean_tta'),
          gv(f'p2_test_clean_imgsz{HIGHRES_IMGSZ}')]

x = np.arange(len(settings)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, b_vals, w, label='Baseline')
ax.bar(x + w/2, p_vals, w, label='P2')
ax.set_xticks(x); ax.set_xticklabels(settings)
ax.set_ylabel('mAP@0.5 (test)'); ax.set_ylim(0, 1.0)
ax.set_title('Inference-time enhancements: TTA + high-resolution')
ax.grid(axis='y', alpha=0.3); ax.legend()
for xi, v in zip(x - w/2, b_vals):
    if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
for xi, v in zip(x + w/2, p_vals):
    if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(DRIVE_ABL_DIR / 'plots' / 'tta_highres_bars.png', dpi=150)
plt.show()